## This code will implement the Seq2seq model

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import pickle
import numpy as np
import torch, numpy as np
from torch.utils.data import Dataset
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader

## Pipeline comparaison

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
with open("/kaggle/input/new-seq2seq/Global_mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

In [4]:
class MMapSeq2Seq(Dataset):
    def __init__(self, prefix=""):
        self.ctx = np.load(prefix+"context_np.npy","r")
        self.x   = np.load(prefix+"X_np.npy","r")
        self.y   = np.load(prefix+"Y_np.npy","r")

        self.ctx_off = np.load(prefix+"context_offset.npy")
        self.ctx_len = np.load(prefix+"context_length.npy")
        self.x_off   = np.load(prefix+"X_offset.npy")
        self.x_len   = np.load(prefix+"X_length.npy")
        self.y_off   = np.load(prefix+"Y_offset.npy")
        self.y_len   = np.load(prefix+"Y_length.npy")

        assert len(self.ctx_len) == len(self.x_len) == len(self.y_len)

    def __len__(self):
        return len(self.ctx_len)

    def __getitem__(self, i):
        # vues memmap -> torch (zéro copie CPU)
        s, L = int(self.ctx_off[i]), int(self.ctx_len[i])
        ctx = torch.from_numpy(self.ctx[s:s+L].astype(np.int32))

        s, L = int(self.x_off[i]), int(self.x_len[i])
        x = torch.from_numpy(self.x[s:s+L].astype(np.int32))

        s, L = int(self.y_off[i]), int(self.y_len[i])
        y = torch.from_numpy(self.y[s:s+L].astype(np.int32))

        return ctx, x, y

In [5]:
def collate(batch):
    ctxs, xs, ys = zip(*batch)  # tuples de 1D tensors CPU
    ctxs = [t.to(torch.long, copy=False) for t in ctxs]
    xs   = [t.to(torch.long, copy=False) for t in xs]
    ys   = [t.to(torch.long, copy=False) for t in ys]

    ctx_pad = torch.nn.utils.rnn.pad_sequence(ctxs, batch_first=True, padding_value=PAD_ID)
    x_pad   = torch.nn.utils.rnn.pad_sequence(xs,   batch_first=True, padding_value=PAD_ID)
    y_pad   = torch.nn.utils.rnn.pad_sequence(ys,   batch_first=True, padding_value=PAD_ID)

    ctx_len = torch.tensor([t.numel() for t in ctxs], dtype=torch.int64)  # CPU Long (pack_padded)
    x_len   = torch.tensor([t.numel() for t in xs],   dtype=torch.int64)

    return (ctx_pad, x_pad, y_pad), (ctx_len, x_len)

### Now, pipeline prep

In [6]:
PAD_ID = 0
ds = MMapSeq2Seq(prefix="/kaggle/input/new-seq2seq/")  # mets des préfixes train/val si tu split offline

In [7]:
train_size = int(0.8 * len(ds))
test_size   = len(ds) - train_size
train_ds, test_ds = random_split(ds, [train_size, test_size], torch.Generator().manual_seed(42))

batch_size = 16

In [8]:
if device == "cpu" :
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, collate_fn=collate) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=collate)
else : 
    train_ld = DataLoader(train_ds, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=4, shuffle=True, drop_last=True, persistent_workers=False, collate_fn=collate) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_ld = DataLoader(test_ds, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=2, shuffle=False, drop_last=True, persistent_workers=False, collate_fn=collate)

List of operations :

- First the entire batch pass through the Encodeur, we extract the h,c from this

- Then we iterate over all the length of the sequence, with the previous input and we use at first the h,c from the Encodeur

- Then we calculate the loss

## Create Seq2seq

In [9]:
class Encodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1, bidirectional=True):
        super(Encodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
#        self.lstm = nn.LSTM(input_size = embedding_dim, 
        self.gru = nn.GRU(input_size = embedding_dim,
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.2,
                            batch_first=True,
                            bidirectional=bidirectional)
        self.dropout = nn.Dropout(0.1)

        if bidirectional:
            self.proj = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, x, length_batch):

        emb = self.embed(x)
        emb_drop = self.dropout(emb)

        packed_x = pack_padded_sequence(emb_drop, length_batch, batch_first=True, enforce_sorted=False)                      
        packed_out, h = self.gru(packed_x)             
        output, length_out = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1)) 
        output = self.dropout(output)

        if self.bidirectional:
            # Project encoder outputs
            out = self.proj(output)  # (B,S,H)

            # Combine last hidden states from both directions
            h = h.view(self.num_layers, self.num_directions, x.size(0), self.hidden_size)
            h = h.transpose(1, 2).contiguous().view(self.num_layers, x.size(0), -1)  # (num_layers,B,2H)
            h = self.proj(h)  # (num_layers,B,H)

        return out, length_out, h
        
#        return output, length_out, h

In [10]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)
        
    def forward(self, query, keys, mask):
        #query (B,H)
        #keys/context_out (B,S,H) where S represent each unit time

        q = self.Wa(query).unsqueeze(1)            # (B,1,H)
        k = self.Ua(keys)                           # (B,S,H)
        scores = self.Va(torch.tanh(q + k))  # (B,S,1)
        scores = scores.squeeze(-1) # (B,S)

        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)         # (B,S)
        context = torch.bmm(weights.unsqueeze(1), keys)  # (B,1,H)

        return context #, weights free some memory

In [11]:
class DecoderAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, masked_mapping, num_layers=1):
        super(DecoderAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.mask = masked_mapping
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(hidden_size)
#        self.lstm = nn.LSTM(input_size = embedding_dim + hidden_size, 
        self.gru = nn.GRU(input_size = embedding_dim + hidden_size,
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            dropout = 0.2,
                            batch_first=True)
        self.final = nn.Linear(hidden_size,vocab_size)
        self.dropout = nn.Dropout(0.1) 

    def forward_step(self,x, h, encod_out,mask_att) : #On each timestep/len of the sentence in the batch
    #Perform everything and then send back to forward
        embedded = self.dropout(self.embed(x)) #(B,E)
        embedded = embedded.unsqueeze(1) #(B,1,E)

        #Use of attention by using the latest hidden
        query = h[-1]
        attn_context = self.attention(query,encod_out,mask_att)

        gru_in = torch.cat([embedded, attn_context], dim=-1) #(B,1,E+H)
        out, h = self.gru(gru_in, h)
        
        logit = self.final(out.squeeze(1))      # (B, vocab_size)
        masked_logit = logit.masked_fill(self.mask, float("-inf"))
        
        return masked_logit, h

    def forward(self, x, encod_out, h, targets, teacher_forcing_ratio, mask_att, loss_fn=None): #On each batch

        batch_size, max_len = targets.size()
        input_x = x[:, 0]
        
        if loss_fn is None :
            all_logits = []
        total_loss = 0.0

        for t in range(0, max_len):
            out, h = self.forward_step(input_x, h, encod_out,mask_att)
            if loss_fn is None :
                all_logits.append(out.unsqueeze(1))
            else : 
                total_loss += loss_fn(out, targets[:, t])

            if torch.rand(1).item() < teacher_forcing_ratio:
                input_x = targets[:, t]   # vérité
            else:
                input_x = out.argmax(dim=-1)  # prédiction

            del out

        if loss_fn is None :
            all_logits = torch.cat(all_logits, dim=1)
            return all_logits
        else : 
            total_loss = total_loss / max_len  
            return total_loss

In [12]:
vocab_size = len(mapping)
embedding_size = 64
hidden_size = 256
num_epoch = 100
accum_steps = 16
nb_step_test = len(test_ld)
nb_step_train = len(train_ld)
nb_update_per_epoch = nb_step_train // accum_steps

#Some of the mapping are only for the encodeur so the decodeur can't produce them, we need to mask them from the loss
mapping_inverse = {i: ch for ch, i in mapping.items()}
masked_mapping = list(mapping_inverse.keys())[117:-1]

mask = torch.zeros(vocab_size, dtype=torch.bool, device=device)
mask[masked_mapping] = True

enco = Encodeur(vocab_size, embedding_size, hidden_size, num_layers=2).to(device)
deco = DecoderAttention(vocab_size, embedding_size, hidden_size, mask,num_layers=2).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=0) #Just ignore the padding

params = list(enco.parameters()) + list(deco.parameters())
opti = torch.optim.AdamW(params, lr=0.0025, betas=(0.9,0.999), weight_decay=1e-4)

sched_warm = torch.optim.lr_scheduler.LinearLR(opti, start_factor=0.2, end_factor=1.0, total_iters=nb_update_per_epoch*4)
sched_post = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opti, T_0=nb_update_per_epoch*20, T_mult=2, eta_min=0.001) #1 epoch => 2 => 4 => 8

## Creation boucle entraînement Decodeur

In [13]:
def token_accuracy(logits, Y, pad_id=0):
    # logits: [B, T, V], Y: [B, T]
    preds = logits.argmax(dim=-1)                 # [B, T]
    mask  = (Y != pad_id)                         # [B, T]
    correct = (preds == Y) & mask
    return correct.sum().float() / mask.sum().float()

def topk_token_accuracy(logits, Y, k=5, pad_id=0):
    """
    logits : (B, T, V)
    Y      : (B, T)
    """
    # tronquer Y si jamais logits et Y diffèrent un peu
    Y = Y[:, :logits.size(1)]

    # top-k indices
    topk = logits.topk(k, dim=-1).indices  # (B, T, k)

    # mask pour ignorer les PAD
    mask = (Y != pad_id)

    # comparaison : Y est-il dans top-k ?
    correct = (topk == Y.unsqueeze(-1)).any(dim=-1) & mask  # (B, T)

    return correct.sum().float() / mask.sum().float()


In [ ]:
l_tot = []
teacher_forcing_ratio = 1
#Early stopping
#early_stopping_count = 0
#patience = 5
best_val = float("inf")
begin_epoch = 0
scaler = torch.amp.GradScaler()
#accum_steps = 16   # number of mini-batches to accumulate
#global_step = 0

accum_steps = 16   # nombre de mini-batches accumulés
global_step = 0

for epoch in range(num_epoch):
    enco.train(); deco.train()

    l_train = 0.0
    l_test = 0.0

    for (context, X, Y), (length_cont,length_text) in iter(train_ld):
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)
        context = context.to(device, non_blocking=True)

        # pas de zero_grad ici → on le fait seulement après accum_steps
        encod_out, length_out, h_enco = enco(context,length_cont)

        arange = torch.arange(context.shape[1], device=length_out.device).unsqueeze(0)  
        mask_attn = (arange < length_out.unsqueeze(1)).to(device)

        with torch.amp.autocast(device_type="cuda"):
            loss = deco(X, encod_out, h_enco, Y, teacher_forcing_ratio , mask_attn, loss_fn)

        # normalisation par accum_steps
        loss = loss / accum_steps

        scaler.scale(loss).backward()

        # clip après backward mais avant step
        if (global_step + 1) % accum_steps == 0:
            scaler.unscale_(opti)
            torch.nn.utils.clip_grad_norm_(list(enco.parameters()) + list(deco.parameters()), 0.25)

            scaler.step(opti)
            scaler.update()
            opti.zero_grad(set_to_none=True)

            step_scheduler = sched_warm if epoch < 2 else sched_post
            step_scheduler.step()


        global_step += 1

        del loss, encod_out, mask_attn
        torch.cuda.empty_cache()

        # scheduler
    if epoch in [20, 40, 80]:
        sched_post.base_lrs[0] *= 0.9
        print(f"Decrease {sched_post.base_lrs[0]}, {sched_post.eta_min}")

#        step_scheduler = sched_warm if epoch < 2 else sched_post
#        step_scheduler.step()

    #Test data part
    
    enco.eval()
    deco.eval()

    acc_sum, tok_sum = 0.0, 0
    topk_acc_sum = 0.0    
    
    with torch.inference_mode():
        with torch.amp.autocast("cuda"):
            for (context, X, Y), (length_cont,length_text) in iter(test_ld) :
                X = X.to(device, non_blocking = True)
                Y = Y.to(device, non_blocking = True)
                context = context.to(device, non_blocking = True)
                
                encod_out, length_out, h_enco = enco(context,length_cont)
                arange = torch.arange(context.shape[1], device=length_out.device).unsqueeze(0)  # (1,S)
                mask_attn = arange < length_out.unsqueeze(1)
                mask_attn = mask_attn.to(device)
                
                logits = deco(X, encod_out, h_enco, Y, 1, mask_attn)
                loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))

                #logits, (h_dec, c_dec) = deco(X,h_dec,c_dec,length_text)
                #loss = loss_fn(logits.reshape(-1,vocab_size),Y.reshape(-1))
                l_test += loss.item()
                acc = token_accuracy(logits, Y).item()
                
                # top-1 accuracy
                acc_sum += ((logits.argmax(-1) == Y) & (Y != 0)).sum().item()
                tok_sum += (Y != 0).sum().item()

                # top-k accuracy (par ex. k=5)
                topk_acc_sum += topk_token_accuracy(logits, Y, k=5).item()

        epoch_token_acc = acc_sum / tok_sum
        epoch_topk_acc  = topk_acc_sum / nb_step_test   # moyenne sur les batches
        print(epoch, np.exp(l_test/nb_step_test), epoch_token_acc, epoch_topk_acc, "\n")
        if np.exp(l_test/nb_step_test) < 4 :
            if begin_epoch == 0 :
                begin_epoch = epoch 
            else :
                teacher_forcing_ratio = max(0.5, 1.0 - (max(0,epoch-begin_epoch)) * 0.01)
        #Record the loss of the epoch
        l_tot.append(l_test); 

        if l_test < best_val :
            best_val = l_test
#            early_stopping_count = 0
            torch.save({
                "epoch": epoch,
                "encoder_state_dict": enco.state_dict(),
                "decoder_state_dict": deco.state_dict(),
                "optimizer_state_dict": opti.state_dict(),
                "scheduler_state_dict": sched_post.state_dict(),
                "val_loss": l_test,
            }, "model1")
        
#        elif l_test >= best_val :
#            early_stopping_count += 1

#        if early_stopping_count == patience :
#            print("Early Stopping")
#            break 

print(f"Liste of offset used : {list_offset}")

0 16.07012634453104 0.23517406360305804 0.5983339712160443 

1 9.702269171328771 0.3412014654465764 0.7216428037083477 

2 7.7858596088998695 0.39980817594280965 0.7562945747594221 

3 6.778987874209783 0.4362380179667765 0.7799293748829343 

4 6.23147907432279 0.4578070718392247 0.7935101548466114 

5 5.85648251043889 0.4765489513804083 0.8037152694999625 

6 5.574200735806836 0.4902052628937551 0.8121442439359262 

7 5.372191118304551 0.4990046338892885 0.8187091202910887 

8 5.210976334610906 0.5075809536555308 0.8223205837634725 

9 5.0863335581824565 0.515772510051246 0.8264229888216071 

10 4.9690792437475695 0.5224250129648527 0.8298016097567497 

11 4.876427089289195 0.5287429390345227 0.8333263670632599 

12 4.798095093578875 0.5322113856499991 0.8366204029923185 

13 4.74048016005733 0.5362987994267583 0.8378404384359307 

14 4.681791712158048 0.5392765333377944 0.8389464726141833 

15 4.634716635175083 0.5436093034667737 0.8406509314108332 

16 4.611352569331443 0.5439940668

1
0 11.675288913783762 0.2950854985143089 0.6734341780344645 

1 6.2799615573336816 0.4495276832721951 0.7939480443795522 

2 5.299127368542268 0.49993981208700006 0.8216034770011902 

3 4.817367883181244 0.5263464669695069 0.836678147315979 

4 4.527002192345504 0.5432624383073892 0.846520314613978 

5 4.309514293531654 0.5575744904617996 0.8526131411393484 

6 4.148890845375624 0.5692382744442122 0.8580587108929952 

7 4.023552679586994 0.5791090921762049 0.8620187739531199 

8 3.9333244365271818 0.5859261652696736 0.8649932245413462 

9 3.8658556585122503 0.5925658424090371 0.8666704297065735 

0.8
10 4.49182240145263 0.5633018455514797 0.8529850840568542 

11 4.34400394865996 0.5739772806467349 0.8572644591331482 

12 4.180594575192007 0.5824289307458866 0.8604991833368937 

13 4.177239180218542 0.584811104986727 0.8617366552352905 

14 4.1234268111513686 0.5878458429158826 0.8622150619824728 

15 4.091573220865865 0.589619802456934 0.8635412851969401 

16 4.034260344477441 0.5929459765964052 0.8646471997102102 

17 4.063368092342488 0.593281761795247 0.8643040060997009 

18 3.997626070910638 0.596373519852508 0.8661218583583832 

19 3.9989107496116905 0.5963164997244027 0.8655415574709574 

0.8
20 3.9852081286348615 0.5968676942960865 0.8658081988493601 

21 3.975800831499961 0.5981474793935593 0.8667225241661072 

Decrease 0.008, 0.001
22 4.119955466897837 0.5841712124379906 0.8623375097910563 

23 4.0597046792878935 0.5880612522887246 0.863156924645106 

24 4.068848392591917 0.5909376009731435 0.8626930912335714 

25 4.082602388897783 0.58859344015104 0.8640171488126119 

26 4.091715972557824 0.5903420574129334 0.8623087108135223 

27 4.021493212121764 0.5928002584912474 0.8647315899531046 

28 3.9955715386370785 0.5949416810800879 0.8665619492530823 

29 4.0023239201591165 0.5962721507358765 0.8654279311498007 

0.8
30 3.9690868099265644 0.5971337882272442 0.8657564520835876 

31 3.9615050963336653 0.598508606871559 0.8656251132488251 

32 3.9182169337612494 0.6006120160416627 0.8675306538740793 

33 3.928434233491954 0.6000608214699789 0.8668514887491862 

34 3.8607318391390777 0.6038241499249235 0.8690848449865977 

35 3.855646754971977 0.6044133579153441 0.8691689471403757 

36 3.8265690640264562 0.6073087133091315 0.870070199171702 

37 3.8491129270652076 0.6073403911580788 0.8699588775634766 

38 3.806081054799216 0.6099886593300768 0.8708654741446177 

39 3.815406428984717 0.6103941357966028 0.8705897132555643 

0.8
40 3.7819245398959094 0.6109389947984972 0.8714823126792908 

41 3.7683137942754854 0.613435209295548 0.8716695706049601 

42 3.787130142761487 0.6126305919322854 0.8715757131576538 

43 3.7297576109716144 0.6157160144197569 0.8727750182151794 

44 3.7203626750267764 0.6161975177237565 0.8728966812292734 

45 3.69458816052927 0.6176927121940712 0.874243418375651 

46 3.6934647229993036 0.6193462959091226 0.8733510375022888 

47 3.674176393063698 0.620474027331648 0.8736821810404459 

48 3.6452689529468647 0.6217601479989103 0.874923586845398 

49 3.649895062520535 0.6224633962455414 0.8748176097869873 

0.8
50 3.6255543464157283 0.6229639062589094 0.8750980695088705 

51 3.61350710120744 0.6242183490772243 0.8760584990183512 

52 3.6045455694574096 0.6259479596297493 0.8760957022507986 

53 3.6152460096294576 0.6255741610121707 0.8760235210259756 

54 3.6052236924464522 0.6267589125628014 0.8763332466284434 

55 3.5961687399444497 0.6263661072358543 0.8767567674318949 

56 3.581494534698449 0.6277345903103796 0.876611590385437 

57 3.5781619114728644 0.628159073486274 0.8774084448814392 

58 3.5701388384334574 0.629293140478589 0.8776156008243561 

59 3.565291782056719 0.6294578652931151 0.8773641387621561 

0.8
60 3.5548772034042764 0.6297239592242728 0.8779175082842509 

61 3.5733844718730663 0.6298189927711149 0.8772278825441996 

Decrease 0.0064, 0.001
62 3.6603389215703777 0.6208034769607005 0.8749801218509674 

63 3.701571370988334 0.6192385912227016 0.8733106354872385 

64 3.7013307440997405 0.6192956113508068 0.8741029302279154 

65 3.6915093756396122 0.6186747255114389 0.8737419346968333 

66 3.702211255745259 0.6195173562934383 0.8735553423563639 

67 3.6843166809010737 0.6219312083832259 0.8749035596847534 

68 3.678375741293826 0.6211519332991212 0.8741554915904999 

69 3.6579619694068333 0.622419047257015 0.8745552599430084 

0.8
70 3.665216885562114 0.6204803629014375 0.874726136525472 

71 3.6590129596284457 0.6221276110466994 0.8756341139475504 

72 3.68981171274146 0.6211329265897528 0.8740512430667877 

73 3.658487681183961 0.6227358257464886 0.8743649820486704 

74 3.6915440673735143 0.6221656244654363 0.8739129702250162 

75 3.6415693353698693 0.6221592888956469 0.876004030307134 

76 3.647018363606229 0.6247061879510134 0.8759126762549082 

77 3.6229626509019037 0.6252637180924866 0.8762900133927664 

78 3.621944251583894 0.6260176508974334 0.8756223122278849 

79 3.5866304062158236 0.6277726037291164 0.8773874541123708 

0.8
80 3.607604148554642 0.6263154226775385 0.8767216602961222 

81 3.5939530607439845 0.6279119862644847 0.8771616419156393 

82 3.616202891657272 0.6281020533581688 0.8760560353597006 

83 3.6430534942786545 0.6272024024480641 0.8750207324822744 

84 3.5789863856812856 0.6291220800942733 0.8768300414085388 

85 3.5701746594174675 0.6306299457041669 0.8776143590609232 

86 3.563166557020617 0.6312571671133244 0.8777105708916982 

87 3.5417099765638964 0.6305349121573248 0.8780483106772105 

88 3.546127301642493 0.6318907240922712 0.8777520855267843 

89 3.5559240802546275 0.6328727374096389 0.8783309559027354 

0.8
90 3.556986035600665 0.6329741065262704 0.877832273642222 

91 3.5338780958373275 0.6335696500864805 0.8784035940965017 

92 3.5448474317827525 0.6345960123923745 0.877895583709081 

93 3.53050672777521 0.6346910459392165 0.8784897228082021 

94 3.548040394887739 0.6357427505242684 0.8775809903939565 

95 3.5319673758178323 0.6355653545701633 0.878033846616745 

96 3.4996577638409856 0.6361038780022681 0.8795545697212219 

97 3.5074182644086758 0.6368007906791098 0.8792359133561453 

98 3.4976039061506343 0.6377321194381617 0.8795585930347443 

99 3.482688878230835 0.6381375959046877 0.8800532321135203 

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
/tmp/ipykernel_36/1792155417.py in <cell line: 0>()
    129 #            break
